# Agent: secuencia, ramas y decisiones · 2.1.4

Un constructor Agent, un Graph. Input value=2 → proceso visible → 6, 7 y 6. El paralelo usa campos distintos y una barrera.

In [2]:
import pydantic
import agentic_systems as toolkit
assert toolkit.__version__ == '2.1.4'

class State(pydantic.BaseModel):
    value: int
    increased: int | None = None
    doubled: int | None = None

def increase(state):
    return {'value': state.value + 1}

def double(state):
    return {'value': state.value * 2}


Secuencia

In [3]:
graph = toolkit.Graph(State)
graph.add_node('increase', increase)
graph.add_node('double', double)
graph.add_edge(toolkit.START, 'increase')
graph.add_edge('increase', 'double')
graph.add_edge('double', toolkit.END)
agent = toolkit.Agent('arithmetic', graph=graph)
result = agent.run({'value': 2})
assert result.ok and result.data['value'] == 6
assert result.check_invariants().ok
print(result.human_results())


arithmetic: COMPLETED
Execution: run-8a8f00dacda0434eb94e5c8b50b7ae65
Input: {"value":2}
Output: {"value":6}
Process:
- Step increase: completed
- Step double: completed


Paralelo: dos ramas y barrera

In [4]:
def left(state):
    return {'increased': state.value + 1}

def right(state):
    return {'doubled': state.value * 2}

def total(state):
    return {'value': state.increased + state.doubled}
graph = toolkit.Graph(State)
graph.add_node('left', left)
graph.add_node('right', right)
graph.add_node('total', total)
graph.add_edge(toolkit.START, 'left')
graph.add_edge(toolkit.START, 'right')
graph.add_edge(['left', 'right'], 'total')
graph.add_edge('total', toolkit.END)
agent = toolkit.Agent('concurrent', graph=graph)
result = agent.run({'value': 2})
assert result.ok and result.data['value'] == 7
assert result.check_invariants().ok
print(result.human_results())


concurrent: COMPLETED
Execution: run-d13a344f514f4e95ba53d111e250f71e
Input: {"value":2}
Output: {"value":7,"increased":3,"doubled":4}
Process:
- Step left: completed
- Step right: completed
- Step total: completed


Decisión: una función normal

In [5]:
graph = toolkit.Graph(State)
graph.add_node('increase', increase)
graph.add_node('double', double)
graph.add_edge(toolkit.START, 'increase')
graph.add_conditional_edges('increase', lambda state: 'double' if state.value >= 0 else toolkit.END, ['double', toolkit.END])
graph.add_edge('double', toolkit.END)
agent = toolkit.Agent('decision', graph=graph)
result = agent.run({'value': 2})
assert result.ok and result.data['value'] == 6
assert result.check_invariants().ok
print(result.human_results())


decision: COMPLETED
Execution: run-dc9d2a7b97e840be86cd116ac39ce115
Input: {"value":2}
Output: {"value":6}
Process:
- Step increase: completed
- Decision increase -> "double"
- Step double: completed